# Notebook 46 — Deriving the Stretching Exponent from Channel Structure

This notebook follows the mechanism established in Notebooks 41–45.

So far, the project found:

- a collapsed variable \(x\),
- a universal response \(S(x) pprox \exp(-a x^b)\),
- failure of naive factorized channel products,
- evidence for an emergent rate picture,
- and a mechanism in which a **distribution of decay rates** generates stretched-exponential behavior.

The next question is:

> Can the fitted stretching exponent \(b\) be related to the structure of the channel mixture itself?

This notebook does that in a controlled way.

## Main goals

1. build synthetic channel mixtures with tunable heterogeneity,
2. generate ensemble decays
   \[
   S(x) = \langle e^{-\Gamma x} angle,
   \]
3. fit the resulting stretched exponent \(b\),
4. measure simple statistics of the underlying rate distribution,
5. test whether \(b\) can be predicted from channel heterogeneity.

This is not yet a first-principles derivation from the full Lindblad generator. Instead, it is a bridge notebook:
it derives how **channel diversity / rate-distribution width** maps onto the observed effective exponent.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit


## Helper laws

In [ ]:
def stretched_law(x, a, b):
    return np.exp(-a * np.power(x, b))

def exp_law(x, a):
    return np.exp(-a * x)


## Ensemble decay from a rate distribution

In [ ]:
def ensemble_decay(rates, x):
    rates = np.asarray(rates, dtype=float)
    return np.array([np.mean(np.exp(-rates * xi)) for xi in x], dtype=float)

def fit_stretched(x, S):
    x = np.clip(np.asarray(x, dtype=float), 1e-12, None)
    S = np.clip(np.asarray(S, dtype=float), 1e-12, 1.0)
    popt, _ = curve_fit(
        stretched_law,
        x,
        S,
        p0=[1.0, 1.0],
        bounds=([0.0, 0.1], [100.0, 5.0]),
        maxfev=50000,
    )
    a, b = [float(v) for v in popt]
    pred = stretched_law(x, a, b)
    mse = float(np.mean((S - pred) ** 2))
    return {"a": a, "b": b, "pred": pred, "mse": mse}

def fit_exponential(x, S):
    x = np.clip(np.asarray(x, dtype=float), 1e-12, None)
    S = np.clip(np.asarray(S, dtype=float), 1e-12, 1.0)
    popt, _ = curve_fit(
        exp_law,
        x,
        S,
        p0=[1.0],
        bounds=([0.0], [100.0]),
        maxfev=50000,
    )
    a = float(popt[0])
    pred = exp_law(x, a)
    mse = float(np.mean((S - pred) ** 2))
    return {"a": a, "pred": pred, "mse": mse}


## Synthetic rate distributions

In [ ]:
rng = np.random.default_rng(42)

def make_gamma_rates(mean_rate, shape_k, n=30000):
    # Gamma(shape=k, scale=theta) with mean = k*theta
    theta = mean_rate / shape_k
    return rng.gamma(shape=shape_k, scale=theta, size=n)

def make_lognormal_rates(mean_rate, sigma, n=30000):
    # choose mu so that lognormal mean matches mean_rate
    mu = np.log(mean_rate) - 0.5 * sigma**2
    return rng.lognormal(mean=mu, sigma=sigma, size=n)

def rate_stats(rates):
    rates = np.asarray(rates, dtype=float)
    mean = float(np.mean(rates))
    std = float(np.std(rates))
    cv = float(std / mean) if mean > 0 else np.nan
    q10, q50, q90 = np.quantile(rates, [0.10, 0.50, 0.90])
    width_ratio = float(q90 / q10) if q10 > 0 else np.nan
    return {
        "mean": mean,
        "std": std,
        "cv": cv,
        "q10": float(q10),
        "q50": float(q50),
        "q90": float(q90),
        "width_ratio": width_ratio,
    }


## Sweep over channel heterogeneity

In [ ]:
x = np.linspace(0.0, 0.30, 240)

results = []

# Family A: gamma-distributed rates, shape controls heterogeneity
mean_rate = 8.0
shape_vals = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]

for k in shape_vals:
    rates = make_gamma_rates(mean_rate=mean_rate, shape_k=k, n=25000)
    S = ensemble_decay(rates, x)
    st = fit_stretched(x[1:], S[1:])  # skip x=0 for fit stability
    ex = fit_exponential(x[1:], S[1:])
    stats = rate_stats(rates)
    results.append({
        "family": "gamma",
        "control": float(k),
        "control_name": "shape_k",
        "rates": rates,
        "S": S,
        "stretched": st,
        "exp": ex,
        **stats,
    })

# Family B: lognormal rates, sigma controls heterogeneity
sigma_vals = [0.10, 0.20, 0.35, 0.50, 0.70, 0.90, 1.10]

for sigma in sigma_vals:
    rates = make_lognormal_rates(mean_rate=mean_rate, sigma=sigma, n=25000)
    S = ensemble_decay(rates, x)
    st = fit_stretched(x[1:], S[1:])
    ex = fit_exponential(x[1:], S[1:])
    stats = rate_stats(rates)
    results.append({
        "family": "lognormal",
        "control": float(sigma),
        "control_name": "sigma",
        "rates": rates,
        "S": S,
        "stretched": st,
        "exp": ex,
        **stats,
    })

print("Total synthetic cases:", len(results))


## Summary table data

In [ ]:
summary_rows = []
for row in results:
    summary_rows.append({
        "family": row["family"],
        "control_name": row["control_name"],
        "control": row["control"],
        "mean_rate": row["mean"],
        "cv": row["cv"],
        "width_ratio": row["width_ratio"],
        "b_fit": row["stretched"]["b"],
        "a_fit": row["stretched"]["a"],
        "mse_stretched": row["stretched"]["mse"],
        "mse_exp": row["exp"]["mse"],
    })

# Print a compact summary
for r in summary_rows:
    print(
        r["family"],
        r["control_name"], f'{r["control"]:.3f}',
        "cv=", f'{r["cv"]:.3f}',
        "width=", f'{r["width_ratio"]:.3f}',
        "b=", f'{r["b_fit"]:.3f}',
        "mse_st=", f'{r["mse_stretched"]:.3e}',
        "mse_exp=", f'{r["mse_exp"]:.3e}',
    )


## Plot representative ensemble decays

In [ ]:
plt.figure(figsize=(8.0, 5.3))

for idx in [0, 3, 7, len(shape_vals), len(shape_vals)+2, len(results)-1]:
    row = results[idx]
    label = f'{row["family"]}: {row["control_name"]}={row["control"]:.2f}, b={row["stretched"]["b"]:.2f}'
    plt.plot(x, row["S"], label=label)

plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Representative ensemble decays from rate mixtures")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Plot stretched fits against ensemble data

In [ ]:
plt.figure(figsize=(8.2, 5.4))

for idx in [0, 3, 7, len(shape_vals), len(shape_vals)+2, len(results)-1]:
    row = results[idx]
    label_data = f'{row["family"]} data ({row["control_name"]}={row["control"]:.2f})'
    label_fit = f'fit b={row["stretched"]["b"]:.2f}'
    plt.plot(x, row["S"], linewidth=2.0, alpha=0.9, label=label_data)
    plt.plot(x[1:], row["stretched"]["pred"], linestyle='--', linewidth=1.6, label=label_fit)

plt.xlabel("x")
plt.ylabel("S(x)")
plt.title("Stretched-exponential fits to ensemble decays")
plt.legend(fontsize=7)
plt.tight_layout()
plt.show()


## Does the stretching exponent track heterogeneity?

In [ ]:
families = sorted(set(r["family"] for r in results))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))

for fam in families:
    fam_rows = [r for r in results if r["family"] == fam]
    controls = np.array([r["control"] for r in fam_rows], dtype=float)
    bvals = np.array([r["stretched"]["b"] for r in fam_rows], dtype=float)
    axes[0].plot(controls, bvals, marker="o", label=fam)

axes[0].set_xlabel("distribution control parameter")
axes[0].set_ylabel("fitted stretching exponent b")
axes[0].set_title("b vs distribution-family control")
axes[0].legend()

for fam in families:
    fam_rows = [r for r in results if r["family"] == fam]
    cvs = np.array([r["cv"] for r in fam_rows], dtype=float)
    bvals = np.array([r["stretched"]["b"] for r in fam_rows], dtype=float)
    axes[1].scatter(cvs, bvals, s=50, label=fam)

axes[1].set_xlabel("coefficient of variation of rates")
axes[1].set_ylabel("fitted stretching exponent b")
axes[1].set_title("b vs rate heterogeneity")
axes[1].legend()

plt.tight_layout()
plt.show()


## Fit a simple empirical law: b as a function of heterogeneity

In [ ]:
# Use CV as the simplest heterogeneity measure
cv_all = np.array([r["cv"] for r in results], dtype=float)
b_all = np.array([r["stretched"]["b"] for r in results], dtype=float)

# Simple empirical model: b ≈ 1 / (1 + alpha * CV^beta)
def b_model(cv, alpha, beta):
    return 1.0 / (1.0 + alpha * np.power(cv, beta))

popt_b, _ = curve_fit(
    b_model,
    cv_all,
    b_all,
    p0=[1.0, 1.0],
    bounds=([0.0, 0.0], [100.0, 10.0]),
    maxfev=50000,
)
alpha_fit, beta_fit = [float(v) for v in popt_b]
b_pred = b_model(cv_all, alpha_fit, beta_fit)
mse_b = float(np.mean((b_all - b_pred) ** 2))

print("Empirical b(CV) fit:")
print("alpha =", alpha_fit)
print("beta  =", beta_fit)
print("MSE   =", mse_b)


## Visualize the b(CV) relation

In [ ]:
cv_grid = np.linspace(np.min(cv_all), np.max(cv_all), 400)

plt.figure(figsize=(7.2, 5.0))
plt.scatter(cv_all, b_all, s=55, alpha=0.8, label="synthetic cases")
plt.plot(cv_grid, b_model(cv_grid, alpha_fit, beta_fit), linewidth=2.5, label="empirical fit")
plt.xlabel("coefficient of variation of Γ")
plt.ylabel("stretched exponent b")
plt.title("Empirical mapping from rate heterogeneity to b")
plt.legend()
plt.tight_layout()
plt.show()


## Compare exponential vs stretched-exponential error

In [ ]:
mse_exp = np.array([r["exp"]["mse"] for r in results], dtype=float)
mse_st = np.array([r["stretched"]["mse"] for r in results], dtype=float)

plt.figure(figsize=(7.4, 5.0))
plt.scatter(mse_exp, mse_st, s=55)
lims = [min(np.min(mse_exp), np.min(mse_st)), max(np.max(mse_exp), np.max(mse_st))]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("MSE of pure exponential fit")
plt.ylabel("MSE of stretched-exponential fit")
plt.title("Stretched fits outperform pure exponentials for heterogeneous rates")
plt.tight_layout()
plt.show()


## Compact summary

In [ ]:
lines = []
lines.append("Deriving the stretching exponent from channel structure")
lines.append("")
lines.append("Synthetic conclusion:")
lines.append("- A broader distribution of decay rates produces a smaller effective exponent b.")
lines.append("- Narrow distributions approach pure exponential behavior (b -> 1).")
lines.append("- Wide distributions produce stronger stretching (b < 1).")
lines.append("")
lines.append(f"Empirical b(CV) fit: b ≈ 1 / (1 + {alpha_fit:.6f} * CV^{beta_fit:.6f})")
lines.append(f"MSE of empirical b(CV) fit = {mse_b:.6e}")
lines.append("")
lines.append("Representative cases:")
for r in summary_rows[:5]:
    lines.append(
        f'- {r["family"]} {r["control_name"]}={r["control"]:.3f}: '
        f'CV={r["cv"]:.3f}, width={r["width_ratio"]:.3f}, '
        f'b={r["b_fit"]:.3f}, '
        f'MSE_exp={r["mse_exp"]:.3e}, MSE_st={r["mse_stretched"]:.3e}'
    )
lines.append("")
lines.append("Interpretation:")
lines.append("- The stretching exponent is not arbitrary; it encodes heterogeneity in the channel-induced rate distribution.")
lines.append("- In the narrow-distribution limit, the system approaches single-rate exponential decay.")
lines.append("- In the broad-distribution limit, the system exhibits stronger stretched-exponential universality.")
lines.append("- This gives a mechanistic route from channel structure to the observed universal exponent.")

print("\n".join(lines))


## Final conclusion

This notebook derives the qualitative origin of the stretching exponent from channel structure.

The main lesson is:

- **single effective rate**  → exponential decay, \(b pprox 1\)
- **distributed rates**     → stretched exponential, \(b < 1\)

So the observed exponent \(b\) can be interpreted as a compact measure of **rate heterogeneity induced by coupled open-system channels**.

This is the natural continuation after Notebook 45 because it moves from “rate distributions can cause stretching” to “the amount of stretching is controlled by the width and diversity of that distribution.”
